In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import json

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)


In [ ]:
customer_df = pd.read_csv(
    "../data/processed/churn_selected_features.csv",
    parse_dates=["snapshot_date"]
)

selected_model = joblib.load(
    "../models/logistic_regression_pipeline.pkl"
)

with open("../models/model_metadata.json", "r") as file:
    model_metadata = json.load(file)

print("Customer dataset shape:", customer_df.shape)
print("Selected model:", model_metadata["selected_model"])

print("\nAvailable snapshot dates:")
print(sorted(customer_df["snapshot_date"].unique()))

In [ ]:
latest_date = customer_df["snapshot_date"].max()

risk_df = customer_df[
    customer_df["snapshot_date"] == latest_date
].copy()

print("Latest snapshot:", latest_date)
print("Customer records:", len(risk_df))
print("Unique customers:", risk_df["customer_id"].nunique())

In [ ]:
feature_columns = (
    model_metadata["numerical_features"]
    + model_metadata["categorical_features"]
)

missing_columns = set(feature_columns) - set(risk_df.columns)

if missing_columns:
    raise ValueError(f"Missing model features: {missing_columns}")

X_risk = risk_df[feature_columns].copy()

print("Feature count:", len(feature_columns))
print("Feature shape:", X_risk.shape)

In [ ]:
risk_df["churn_probability"] = (
    selected_model.predict_proba(X_risk)[:, 1]
)

print("Risk scores generated successfully.")

print("\nChurn probability distribution:")
print(risk_df["churn_probability"].describe().round(4))

In [ ]:
risk_categories = ["Low Risk", "Medium Risk", "High Risk"]

risk_df["risk_category"] = pd.qcut(
    risk_df["churn_probability"].rank(method="first"),
    q=3,
    labels=risk_categories
)

risk_distribution = risk_df["risk_category"].value_counts().reindex(
    risk_categories
)

print("Customer Risk Segmentation:")
print(risk_distribution)

In [ ]:
plt.figure(figsize=(8, 5))

sns.barplot(
    x=risk_distribution.index,
    y=risk_distribution.values
)

plt.xlabel("Risk Category")
plt.ylabel("Number of Customers")
plt.title("Customer Risk Distribution")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(9, 5))

sns.histplot(
    data=risk_df,
    x="churn_probability",
    bins=30,
    kde=True
)

plt.xlabel("Predicted Churn Probability")
plt.ylabel("Number of Customers")
plt.title("Customer Churn Probability Distribution")
plt.tight_layout()
plt.show()

In [ ]:
transactions_df = pd.read_csv(
    "../data/raw/transactions_data.csv",
    parse_dates=["date"]
)

transactions_df = transactions_df.drop_duplicates().copy()

print("Transactions shape:", transactions_df.shape)

print("\nTransaction date range:")
print(
    transactions_df["date"].min(),
    "to",
    transactions_df["date"].max()
)

print("\nTransaction columns:")
print(transactions_df.columns.tolist())

print("\nMissing values:")
print(transactions_df[["customer_id", "date", "amount"]].isnull().sum())

In [ ]:
required_columns = ["customer_id", "date", "amount"]

missing_columns = set(required_columns) - set(transactions_df.columns)

if missing_columns:
    raise ValueError(f"Missing transaction columns: {missing_columns}")

transactions_df = transactions_df.dropna(
    subset=required_columns
)

In [ ]:
snapshot_date = latest_date
start_date = snapshot_date - pd.Timedelta(days=90)

historical_transactions = transactions_df[
    (transactions_df["date"] > start_date) &
    (transactions_df["date"] <= snapshot_date)
].copy()

customer_transaction_value = historical_transactions.groupby(
    "customer_id"
).agg(
    historical_transaction_value_90d=("amount", "sum"),
    historical_transaction_count_90d=("customer_id", "size")
).reset_index()

risk_df = risk_df.merge(
    customer_transaction_value,
    on="customer_id",
    how="left",
    validate="one_to_one"
)

risk_df[
    [
        "historical_transaction_value_90d",
        "historical_transaction_count_90d"
    ]
] = risk_df[
    [
        "historical_transaction_value_90d",
        "historical_transaction_count_90d"
    ]
].fillna(0)

print("Historical transaction aggregation completed.")
print("Historical period:", start_date, "to", snapshot_date)
print("Risk dataset shape:", risk_df.shape)

print("\nHistorical transaction value:")
print(
    risk_df["historical_transaction_value_90d"]
    .describe()
    .round(2)
)

In [ ]:
risk_summary = risk_df.groupby(
    "risk_category",
    observed=True
).agg(
    customer_count=("customer_id", "count"),
    average_churn_score=("churn_probability", "mean"),
    historical_transaction_value=("historical_transaction_value_90d", "sum"),
    average_transaction_value=("historical_transaction_value_90d", "mean"),
    total_transactions=("historical_transaction_count_90d", "sum")
).reset_index()

risk_summary["customer_percentage"] = (
    risk_summary["customer_count"] / len(risk_df) * 100
)

total_transaction_value = risk_summary[
    "historical_transaction_value"
].sum()

if total_transaction_value != 0:
    risk_summary["transaction_value_percentage"] = (
        risk_summary["historical_transaction_value"]
        / total_transaction_value * 100
    )
else:
    risk_summary["transaction_value_percentage"] = 0

print("Risk Category Summary:")
print(risk_summary.round(2).to_string(index=False))

In [ ]:
if risk_df["customer_id"].duplicated().any():
    raise ValueError("Duplicate customers found in the latest snapshot.")

In [ ]:
risk_df["risk_weighted_transaction_value"] = (
    risk_df["historical_transaction_value_90d"]
    * risk_df["churn_probability"]
)

risk_weighted_summary = risk_df.groupby(
    "risk_category",
    observed=True
).agg(
    risk_weighted_exposure=("risk_weighted_transaction_value", "sum")
).reset_index()

total_exposure = risk_weighted_summary[
    "risk_weighted_exposure"
].sum()

if total_exposure != 0:
    risk_weighted_summary["exposure_percentage"] = (
        risk_weighted_summary["risk_weighted_exposure"]
        / total_exposure * 100
    )
else:
    risk_weighted_summary["exposure_percentage"] = 0

print("Risk-Weighted Transaction Exposure:")
print(risk_weighted_summary.round(2).to_string(index=False))

In [ ]:
plt.figure(figsize=(8, 5))

sns.barplot(
    data=risk_weighted_summary,
    x="risk_category",
    y="risk_weighted_exposure"
)

plt.xlabel("Risk Category")
plt.ylabel("Risk-Weighted Transaction Value")
plt.title("Risk-Weighted Exposure by Customer Category")
plt.tight_layout()
plt.show()

In [ ]:
final_risk_summary = risk_df.groupby(
    "risk_category",
    observed=True
).agg(
    customer_count=("customer_id", "count"),
    average_churn_score=("churn_probability", "mean"),
    historical_transaction_value=("historical_transaction_value_90d", "sum"),
    risk_weighted_exposure=("risk_weighted_transaction_value", "sum")
).reset_index()

final_risk_summary["customer_percentage"] = (
    final_risk_summary["customer_count"] /
    len(risk_df) * 100
)

total_exposure = final_risk_summary[
    "risk_weighted_exposure"
].sum()

if total_exposure != 0:
    final_risk_summary["exposure_percentage"] = (
        final_risk_summary["risk_weighted_exposure"]
        / total_exposure * 100
    )
else:
    final_risk_summary["exposure_percentage"] = 0

final_risk_summary = final_risk_summary.round(2)

print("Final Risk Intelligence Summary:")
print(final_risk_summary.to_string(index=False))